# Phase 9 (step 1) — Quantization-aware training: compatibility test only (no training)
* Goal: check that the merged teacher (Model E) and the merged KD student can be prepared for QAT with
  TensorFlow Model Optimization (tfmot), before any training is run.
* tfmot 0.8.0 "forces users to use the keras v2 version" (tfmot v0.8.0 release notes), so the models, saved with
  Keras 3, are rebuilt with the same architecture in tf_keras (Keras 2) and their weights copied layer by layer.
* Checks: (1) the rebuilt Keras 2 model gives the same predictions as the saved Keras 3 model;
  (2) tfmot.quantization.keras.quantize_model accepts every layer (Conv1D, BatchNormalization, ECA: GAP, Reshape,
  Multiply; skip branch: Flatten, Dense, Concatenate; Dropout); (3) the QAT model converts to a full-integer
  INT8 TFLite model. QAT itself follows Jacob et al. (2018); training is a separate step after approval.
* Runs on a CPU runtime in a few minutes. Writes nothing to the thesis outputs.

In [ ]:
# %% Cell 0 — mount Google Drive and install tf_keras + tfmot (Keras 2 must be selected before TensorFlow is imported)
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, importlib.metadata as md
tfv = md.version('tensorflow'); major_minor = '.'.join(tfv.split('.')[:2])
subprocess.run(f'pip -q install "tf_keras~={major_minor}.0" tensorflow-model-optimization', shell=True, check=True)
os.environ['TF_USE_LEGACY_KERAS'] = '1'
print('TensorFlow', tfv, '| tf_keras', md.version('tf_keras'), '| tfmot', md.version('tensorflow-model-optimization'))

In [ ]:
# %% Cell 1 — rebuild Model E and the student in Keras 2, copy the weights
import time, numpy as np
import tensorflow as tf
import tf_keras as k2                      # Keras 2 (for tfmot)
import keras as k3                         # Keras 3 (to read the saved models)
import tensorflow_model_optimization as tfmot
from sklearn.metrics import f1_score

FEAT  = '/content/drive/MyDrive/Thesis/outputs_v3/features_v3/config1_merged/'
FINAL = '/content/drive/MyDrive/Thesis/outputs_v3/final_v1/models/'
KD    = '/content/drive/MyDrive/Thesis/outputs_v3/kd_v1/models/'
N_CLASSES, N_CHECK, N_REP, SEED = 8, 20000, 500, 42
L = k2.layers

def eca(x):
    c = x.shape[-1]
    w = L.GlobalAveragePooling1D()(x); w = L.Reshape((c, 1))(w)
    w = L.Conv1D(1, 3, padding='same', use_bias=False, activation='sigmoid')(w); w = L.Reshape((1, c))(w)
    return L.Multiply()([x, w])

def build_e(n_features, div=1):
    """Model E (identical to Phases 5 and 7): div = 1 teacher, div = 4 student."""
    inp = k2.Input((n_features, 1))
    x = L.Conv1D(64 // div, 3, padding='same', activation='relu')(inp); x = L.BatchNormalization()(x); x = eca(x)
    x = L.Conv1D(128 // div, 3, padding='same', activation='relu')(x); x = L.BatchNormalization()(x); x = eca(x)
    z = L.GlobalAveragePooling1D()(x)
    s = L.Dense(128 // div, activation='relu')(L.Flatten()(inp))
    z = L.Concatenate()([z, s]); z = L.Dense(64 // div, activation='relu')(z); z = L.Dropout(0.3)(z)
    return k2.Model(inp, L.Dense(N_CLASSES, activation='softmax')(z))

def copy_weights(src, dst):
    a = [l for l in src.layers if l.get_weights()]; b = [l for l in dst.layers if l.get_weights()]
    assert len(a) == len(b), f'{len(a)} vs {len(b)} layers with weights'
    for la, lb in zip(a, b):
        assert type(la).__name__ == type(lb).__name__, (la.name, lb.name)
        assert [w.shape for w in la.get_weights()] == [w.shape for w in lb.get_weights()], (la.name, lb.name)
        lb.set_weights(la.get_weights())
    return len(a)

Xte = np.load(FEAT + 'X_test.npy', mmap_mode='r'); yte = np.load(FEAT + 'y_test.npy')
idx = np.random.default_rng(SEED).choice(len(yte), N_CHECK, replace=False); idx.sort()
Xc = np.asarray(Xte[idx], np.float32)[..., None]; yc = yte[idx]
Xtr = np.load(FEAT + 'X_train.npy', mmap_mode='r')
rep = np.asarray(Xtr[np.sort(np.random.default_rng(SEED).choice(len(Xtr), N_REP, replace=False))], np.float32)[..., None]
print(f'check set {N_CHECK:,} test records | representative set {N_REP} training records')

In [ ]:
# %% Cell 2 — the three checks for the merged teacher and student
for name, path, div in [('teacher_merged', FINAL + 'final_config1_merged.keras', 1),
                        ('student_kd_merged', KD + 'kd_config1_merged.keras', 4)]:
    print(f'\n===== {name} =====')
    m3 = k3.models.load_model(path, compile=False)
    m2 = build_e(Xc.shape[1], div); n = copy_weights(m3, m2)
    p3, p2 = m3.predict(Xc, batch_size=4096, verbose=0), m2.predict(Xc, batch_size=4096, verbose=0)
    print(f'(1) weights copied for {n} layers | max |difference| in probabilities {np.abs(p3 - p2).max():.2e} | '
          f'same predicted class {100 * (p3.argmax(1) == p2.argmax(1)).mean():.2f}%')
    try:
        q = tfmot.quantization.keras.quantize_model(m2)
        print(f'(2) quantize_model: OK ({len(q.layers)} layers)')
    except Exception as e:
        print('(2) quantize_model FAILED:', type(e).__name__, str(e)[:400]); continue
    pq = q.predict(Xc, batch_size=4096, verbose=0).argmax(1)
    print(f'    before any QAT training: Macro F1 float {f1_score(yc, p2.argmax(1), average="macro"):.4f} | '
          f'with simulated INT8 {f1_score(yc, pq, average="macro"):.4f} | same class {100 * (pq == p2.argmax(1)).mean():.2f}%')
    try:
        conv = tf.lite.TFLiteConverter.from_keras_model(q)
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        conv.representative_dataset = lambda: ([rep[i:i + 1]] for i in range(len(rep)))
        conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        conv.inference_input_type = tf.int8; conv.inference_output_type = tf.int8
        b = conv.convert(); print(f'(3) INT8 TFLite conversion: OK ({len(b) / 1024:.1f} KB)')
    except Exception as e:
        print('(3) INT8 conversion FAILED:', type(e).__name__, str(e)[:400])
print('\ncompatibility test: done')